# ASR

Launch this notebook with the `ddpm` kernel. Reports two attack-success metrics for every PIPE-FE scheme, both **in-domain** (`celeba`) and **cross-dataset** (`lfw`, `casia_webface`):

- **L2 ASR**: % of test samples whose reconstructed face matches the original under Facenet512 `euclidean_l2` distance `< 1.1027`.
- **Rep ASR**: % of samples for which the reconstruction recovers `>= thr` protected blocks of the FE codeword (per-scheme `thr`). `unprotected` has no FE secret, so its Rep ASR is defined as its L2 ASR.

Each scheme is read at the checkpoint for which cross-dataset results exist (`ckpt_025000` for unprotected / e8, `ckpt_050000` for the leech / ours schemes). Thresholds and checkpoints match `ckpt_vs_asr.py`.

In [ ]:
import numpy as np
import pandas as pd
import os

In [7]:
# Facenet512 euclidean_l2 verification threshold (scripts/evaluation.py)
L2_THRESHOLD = 1.1027
model = 'facenet'
datasets = ['celeba', 'casia_webface', 'lfw']   # celeba = in-domain, lfw / casia_webface = cross-dataset

# (scheme key, display name, checkpoint, protected-blocks file [None => no FE secret], rep block threshold)
SCHEMES = [
    ('unprotected',               'Unprotected',                'ckpt_025000', None,                                                  None),
    ('facialfe_e8',               'Facial-FE e8',               'ckpt_025000', 'test_protected_blocks_e8.npy',                         64),
    ('facialfe_leech',            'Facial-FE leech',            'ckpt_050000', 'test_protected_blocks_facialfe_leech.npy',             21),
    ('facialfe_leech_randomised', 'Facial-FE leech randomized', 'ckpt_050000', 'test_protected_blocks_facialfe_leech_randomised.npy', 21),
    ('ourfe',                     'Snapper-FE (ours)',          'ckpt_050000', 'test_protected_blocks_ourfe.npy',                       4),
]

def result_dir(scheme, ckpt, dataset):
    """In-domain celeba -> results/<ckpt>/...; cross-dataset -> results/crossdataset/<dataset>/<ckpt>/...
    The inversion model is always celeba/facenet-trained, hence the trailing 'celeba/<model>'."""
    if dataset == 'celeba':
        return f'results/{ckpt}/{scheme}/celeba/{model}'
    return f'results/crossdataset/{dataset}/{ckpt}/{scheme}/celeba/{model}'

def l2_asr(d):
    """% of samples with reconstructed-vs-original Facenet512 distance < threshold."""
    p = f'{d}/test_distances.npy'
    return float(np.mean(np.load(p) < L2_THRESHOLD)) * 100.0 if os.path.exists(p) else np.nan

def rep_asr(d, blk_file, thr, l2):
    """% of samples recovering >= thr protected blocks; unprotected (blk_file=None) => its L2 ASR."""
    if blk_file is None:
        return l2
    p = f'{d}/{blk_file}'
    return float(np.mean(np.load(p) >= thr)) * 100.0 if os.path.exists(p) else np.nan

In [8]:
# Compute L2 + Rep ASR for every (scheme, dataset)
rows = []
for scheme, disp, ckpt, blk_file, thr in SCHEMES:
    for ds in datasets:
        d = result_dir(scheme, ckpt, ds)
        l2 = l2_asr(d)
        rep = rep_asr(d, blk_file, thr, l2)
        rows.append({'scheme': disp, 'dataset': ds, 'checkpoint': ckpt,
                     'L2 ASR (%)': round(l2, 2), 'Rep ASR (%)': round(rep, 2)})
df = pd.DataFrame(rows)

In [9]:
# Cross-dataset L2 (reconstruction) ASR  -- rows = scheme, cols = dataset
order = [s[1] for s in SCHEMES]
l2_table = df.pivot(index='scheme', columns='dataset', values='L2 ASR (%)').reindex(index=order, columns=datasets)
print(f'L2 reconstruction ASR (%)  [Facenet512 euclidean_l2 < {L2_THRESHOLD}]')
l2_table

L2 reconstruction ASR (%)  [Facenet512 euclidean_l2 < 1.1027]


dataset,celeba,casia_webface,lfw
scheme,,,
Unprotected,100.0,100.0,99.9
Facial-FE e8,99.8,99.3,99.7
Facial-FE leech,65.1,61.6,65.1
Facial-FE leech randomized,66.5,61.4,63.7
Snapper-FE (ours),0.4,1.1,0.6


In [10]:
# Cross-dataset Rep (recovered-secret) ASR  -- rows = scheme, cols = dataset
rep_table = df.pivot(index='scheme', columns='dataset', values='Rep ASR (%)').reindex(index=order, columns=datasets)
print('Rep / recovered-secret ASR (%)  [reconstruction recovers >= per-scheme block threshold]')
rep_table

Rep / recovered-secret ASR (%)  [reconstruction recovers >= per-scheme block threshold]


dataset,celeba,casia_webface,lfw
scheme,,,
Unprotected,100.0,100.0,99.9
Facial-FE e8,99.6,99.5,99.9
Facial-FE leech,72.9,68.8,72.7
Facial-FE leech randomized,73.8,68.5,71.6
Snapper-FE (ours),3.7,4.0,3.6
